<a href="https://colab.research.google.com/github/keshav20004/Digit-classification-/blob/main/Digit_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DigitCNN(nn.Module):
    def __init__(self):
        super(DigitCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, 2, padding=1)   # (1,28,28) → (32,28,28)
        self.conv2 = nn.Conv2d(32, 64, 2, padding=1)  # (64,28,28)

        self.pool = nn.MaxPool2d(2,2)                 # halves size

        self.fc1 = nn.Linear(64*7*7, 128)             # FIXED
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # → (32,14,14)
        x = self.pool(F.relu(self.conv2(x)))  # → (64,7,7)

        x = x.view(x.size(0), -1)             # flatten

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DigitCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.14603930908932425
Epoch 2, Loss: 0.04269996657471127
Epoch 3, Loss: 0.02959585934470935
Epoch 4, Loss: 0.02089145978816557
Epoch 5, Loss: 0.016672299345881252
